In [1]:
import duckdb

In [8]:
con = duckdb.connect("yellow.duckdb")

In [9]:
con.sql("CREATE TABLE trips_raw AS SELECT * FROM 'yellow_tripdata_2025-01.parquet'")

In [10]:
con.sql("SELECT COUNT(*) AS total_rows FROM trips_raw").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    3475226 │
└────────────┘



In [11]:
con.sql("SELECT * FROM trips_raw LIMIT 3").show()

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │       double       │
├──────

In [23]:
con.sql("""
    SELECT
  date_trunc('hour', tpep_pickup_datetime) AS hour,
  COUNT(*) AS trips
FROM trips_raw
WHERE tpep_pickup_datetime >= '2025-01-01'
  AND tpep_pickup_datetime <  '2025-02-01'
GROUP BY hour
ORDER BY hour;

""").show()

┌─────────────────────┬───────┐
│        hour         │ trips │
│      timestamp      │ int64 │
├─────────────────────┼───────┤
│ 2025-01-01 00:00:00 │  7344 │
│ 2025-01-01 01:00:00 │  8468 │
│ 2025-01-01 02:00:00 │  7257 │
│ 2025-01-01 03:00:00 │  4915 │
│ 2025-01-01 04:00:00 │  2918 │
│ 2025-01-01 05:00:00 │  1429 │
│ 2025-01-01 06:00:00 │  1220 │
│ 2025-01-01 07:00:00 │  1116 │
│ 2025-01-01 08:00:00 │  1116 │
│ 2025-01-01 09:00:00 │  1524 │
│          ·          │    ·  │
│          ·          │    ·  │
│          ·          │    ·  │
│ 2025-01-31 14:00:00 │  7177 │
│ 2025-01-31 15:00:00 │  7849 │
│ 2025-01-31 16:00:00 │  7818 │
│ 2025-01-31 17:00:00 │ 11173 │
│ 2025-01-31 18:00:00 │ 11874 │
│ 2025-01-31 19:00:00 │ 10191 │
│ 2025-01-31 20:00:00 │  7298 │
│ 2025-01-31 21:00:00 │  7201 │
│ 2025-01-31 22:00:00 │  8569 │
│ 2025-01-31 23:00:00 │  8360 │
├─────────────────────┴───────┤
│ 744 rows          2 columns │
│ (20 shown)                  │
└─────────────────────────────┘



In [13]:
con.sql("""
CREATE OR REPLACE VIEW v_time_geo AS
SELECT
  date_trunc('hour', tpep_pickup_datetime) AS pickup_hour,
  EXTRACT(hour FROM tpep_pickup_datetime)   AS hour_of_day,
  EXTRACT(dow  FROM tpep_pickup_datetime)   AS dow,  -- 0=Sunday … 6=Saturday
  PULocationID,
  DOLocationID
FROM trips_raw
""")


In [27]:
con.sql("SELECT * FROM v_time_geo limit 5")

┌─────────────────────┬─────────────┬───────┬──────────────┬──────────────┐
│     pickup_hour     │ hour_of_day │  dow  │ PULocationID │ DOLocationID │
│      timestamp      │    int64    │ int64 │    int32     │    int32     │
├─────────────────────┼─────────────┼───────┼──────────────┼──────────────┤
│ 2025-01-01 00:00:00 │           0 │     3 │          229 │          237 │
│ 2025-01-01 00:00:00 │           0 │     3 │          236 │          237 │
│ 2025-01-01 00:00:00 │           0 │     3 │          141 │          141 │
│ 2025-01-01 00:00:00 │           0 │     3 │          244 │          244 │
│ 2025-01-01 00:00:00 │           0 │     3 │          244 │          116 │
└─────────────────────┴─────────────┴───────┴──────────────┴──────────────┘

In [14]:
con.sql("""
SELECT pickup_hour, COUNT(*) AS trips
FROM v_time_geo
GROUP BY pickup_hour
ORDER BY pickup_hour
""").show()

┌─────────────────────┬───────┐
│     pickup_hour     │ trips │
│      timestamp      │ int64 │
├─────────────────────┼───────┤
│ 2024-12-31 20:00:00 │     3 │
│ 2024-12-31 21:00:00 │     3 │
│ 2024-12-31 23:00:00 │    15 │
│ 2025-01-01 00:00:00 │  7344 │
│ 2025-01-01 01:00:00 │  8468 │
│ 2025-01-01 02:00:00 │  7257 │
│ 2025-01-01 03:00:00 │  4915 │
│ 2025-01-01 04:00:00 │  2918 │
│ 2025-01-01 05:00:00 │  1429 │
│ 2025-01-01 06:00:00 │  1220 │
│          ·          │    ·  │
│          ·          │    ·  │
│          ·          │    ·  │
│ 2025-01-31 15:00:00 │  7849 │
│ 2025-01-31 16:00:00 │  7818 │
│ 2025-01-31 17:00:00 │ 11173 │
│ 2025-01-31 18:00:00 │ 11874 │
│ 2025-01-31 19:00:00 │ 10191 │
│ 2025-01-31 20:00:00 │  7298 │
│ 2025-01-31 21:00:00 │  7201 │
│ 2025-01-31 22:00:00 │  8569 │
│ 2025-01-31 23:00:00 │  8360 │
│ 2025-02-01 00:00:00 │     1 │
├─────────────────────┴───────┤
│ 748 rows          2 columns │
│ (20 shown)                  │
└─────────────────────────────┘



In [15]:
con.sql("""
SELECT PULocationID, COUNT(*) AS trips
FROM v_time_geo
GROUP BY PULocationID
ORDER BY trips DESC
LIMIT 20
""").show()

┌──────────────┬────────┐
│ PULocationID │ trips  │
│    int32     │ int64  │
├──────────────┼────────┤
│          161 │ 169977 │
│          237 │ 163703 │
│          236 │ 155647 │
│          132 │ 146137 │
│          230 │ 125829 │
│          186 │ 119131 │
│          162 │ 117930 │
│          142 │ 110585 │
│          239 │  96614 │
│          163 │  95906 │
│          234 │  95896 │
│          170 │  95636 │
│           68 │  91241 │
│          138 │  89658 │
│           48 │  84137 │
│          141 │  81661 │
│           79 │  81576 │
│          249 │  77355 │
│          164 │  76066 │
│          140 │  75093 │
├──────────────┴────────┤
│ 20 rows     2 columns │
└───────────────────────┘



In [18]:
con.sql("""
SELECT pickup_hour, PULocationID, COUNT(*) AS trips
FROM v_time_geo
GROUP BY pickup_hour, PULocationID
ORDER BY pickup_hour, trips DESC
""").show()


┌─────────────────────┬──────────────┬───────┐
│     pickup_hour     │ PULocationID │ trips │
│      timestamp      │    int32     │ int64 │
├─────────────────────┼──────────────┼───────┤
│ 2024-12-31 20:00:00 │          246 │     1 │
│ 2024-12-31 20:00:00 │          249 │     1 │
│ 2024-12-31 20:00:00 │           48 │     1 │
│ 2024-12-31 21:00:00 │          179 │     1 │
│ 2024-12-31 21:00:00 │           42 │     1 │
│ 2024-12-31 21:00:00 │          141 │     1 │
│ 2024-12-31 23:00:00 │          114 │     2 │
│ 2024-12-31 23:00:00 │          229 │     2 │
│ 2024-12-31 23:00:00 │          132 │     2 │
│ 2024-12-31 23:00:00 │          163 │     1 │
│          ·          │           ·  │     · │
│          ·          │           ·  │     · │
│          ·          │           ·  │     · │
│ 2025-01-04 09:00:00 │          205 │     1 │
│ 2025-01-04 09:00:00 │           85 │     1 │
│ 2025-01-04 09:00:00 │          241 │     1 │
│ 2025-01-04 09:00:00 │           47 │     1 │
│ 2025-01-04 

In [19]:
con.sql("""
WITH hourly AS (
  SELECT PULocationID,
         pickup_hour,
         COUNT(*) AS trips
  FROM v_time_geo
  GROUP BY PULocationID, pickup_hour
)
SELECT PULocationID, pickup_hour AS peak_hour, trips
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY PULocationID ORDER BY trips DESC) AS rn
  FROM hourly
)
WHERE rn = 1
ORDER BY trips DESC
""").show()


┌──────────────┬─────────────────────┬───────┐
│ PULocationID │      peak_hour      │ trips │
│    int32     │      timestamp      │ int64 │
├──────────────┼─────────────────────┼───────┤
│           79 │ 2025-01-26 01:00:00 │   950 │
│          161 │ 2025-01-23 18:00:00 │   891 │
│          230 │ 2025-01-16 21:00:00 │   708 │
│          249 │ 2025-01-26 00:00:00 │   684 │
│          162 │ 2025-01-23 18:00:00 │   651 │
│          142 │ 2025-01-30 21:00:00 │   635 │
│          236 │ 2025-01-23 15:00:00 │   608 │
│          237 │ 2025-01-23 21:00:00 │   602 │
│          148 │ 2025-01-26 01:00:00 │   579 │
│          132 │ 2025-01-20 22:00:00 │   547 │
│            · │          ·          │     · │
│            · │          ·          │     · │
│            · │          ·          │     · │
│           30 │ 2025-01-20 00:00:00 │     1 │
│           44 │ 2025-01-15 19:00:00 │     1 │
│           27 │ 2025-01-20 10:00:00 │     1 │
│          206 │ 2025-01-25 07:00:00 │     1 │
│           2

In [20]:
con.sql("""
SELECT
  hour_of_day,
  CASE WHEN dow IN (6,0) THEN 'weekend' ELSE 'weekday' END AS day_type,
  COUNT(*) AS trips
FROM v_time_geo
GROUP BY hour_of_day, day_type
ORDER BY day_type, hour_of_day
""").show()


┌─────────────┬──────────┬────────┐
│ hour_of_day │ day_type │ trips  │
│    int64    │ varchar  │ int64  │
├─────────────┼──────────┼────────┤
│           0 │ weekday  │  43024 │
│           1 │ weekday  │  25099 │
│           2 │ weekday  │  16501 │
│           3 │ weekday  │  11618 │
│           4 │ weekday  │  10865 │
│           5 │ weekday  │  18269 │
│           6 │ weekday  │  43186 │
│           7 │ weekday  │  92436 │
│           8 │ weekday  │ 125463 │
│           9 │ weekday  │ 117935 │
│           · │    ·     │    ·   │
│           · │    ·     │    ·   │
│           · │    ·     │    ·   │
│          14 │ weekend  │  53146 │
│          15 │ weekend  │  53513 │
│          16 │ weekend  │  55364 │
│          17 │ weekend  │  55675 │
│          18 │ weekend  │  56383 │
│          19 │ weekend  │  51474 │
│          20 │ weekend  │  42732 │
│          21 │ weekend  │  43732 │
│          22 │ weekend  │  45107 │
│          23 │ weekend  │  43811 │
├─────────────┴──────────┴──

In [21]:
con.sql("""
SELECT PULocationID, DOLocationID, COUNT(*) AS trips
FROM v_time_geo
GROUP BY PULocationID, DOLocationID
ORDER BY trips DESC
LIMIT 30
""").show()


┌──────────────┬──────────────┬───────┐
│ PULocationID │ DOLocationID │ trips │
│    int32     │    int32     │ int64 │
├──────────────┼──────────────┼───────┤
│          237 │          236 │ 24839 │
│          236 │          237 │ 21843 │
│          236 │          236 │ 18221 │
│          237 │          237 │ 17441 │
│          161 │          237 │ 11463 │
│          237 │          161 │ 10168 │
│          161 │          236 │  9845 │
│          239 │          238 │  9595 │
│          142 │          239 │  9141 │
│          239 │          142 │  8844 │
│           ·  │           ·  │    ·  │
│           ·  │           ·  │    ·  │
│           ·  │           ·  │    ·  │
│          263 │          236 │  7421 │
│          186 │          161 │  7172 │
│          230 │          186 │  7168 │
│          142 │          238 │  7025 │
│          237 │          142 │  7018 │
│          141 │          237 │  7004 │
│          239 │          236 │  6981 │
│          162 │          237 │  6955 │


In [22]:
con.sql("""
SELECT
  date_trunc('hour', tpep_pickup_datetime) AS hr,
  AVG(EXTRACT(epoch FROM (tpep_dropoff_datetime - tpep_pickup_datetime))/60) AS avg_duration_min
FROM trips_raw
GROUP BY hr
ORDER BY hr
""").show()


┌─────────────────────┬────────────────────┐
│         hr          │  avg_duration_min  │
│      timestamp      │       double       │
├─────────────────────┼────────────────────┤
│ 2024-12-31 20:00:00 │  19.42777777777778 │
│ 2024-12-31 21:00:00 │              10.35 │
│ 2024-12-31 23:00:00 │  17.57222222222222 │
│ 2025-01-01 00:00:00 │ 16.844771241830088 │
│ 2025-01-01 01:00:00 │  16.60550110218863 │
│ 2025-01-01 02:00:00 │ 16.020745487115878 │
│ 2025-01-01 03:00:00 │ 14.192197355035578 │
│ 2025-01-01 04:00:00 │ 14.511566141192617 │
│ 2025-01-01 05:00:00 │ 13.811476557032904 │
│ 2025-01-01 06:00:00 │ 15.099849726775952 │
│          ·          │          ·         │
│          ·          │          ·         │
│          ·          │          ·         │
│ 2025-01-31 15:00:00 │ 18.084974731388282 │
│ 2025-01-31 16:00:00 │ 17.942380830561998 │
│ 2025-01-31 17:00:00 │  16.44147349264594 │
│ 2025-01-31 18:00:00 │ 15.011984167087776 │
│ 2025-01-31 19:00:00 │ 14.726536486442289 │
│ 2025-01-